#### Data Extraction & Profiling

In [255]:
import pandas as pd
df = pd.read_csv('dirty_cafe_sales.csv')

In [256]:
df

,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
0,TXN_1961373,Coffee,2,2.0,4.0,Credit Card,Takeaway,2023-09-08
1,TXN_4977031,Cake,4,3.0,12.0,Cash,In-store,2023-05-16
2,TXN_4271903,Cookie,4,1.0,ERROR,Credit Card,In-store,2023-07-19
3,TXN_7034554,Salad,2,5.0,10.0,UNKNOWN,UNKNOWN,2023-04-27
4,TXN_3160411,Coffee,2,2.0,4.0,Digital Wallet,In-store,2023-06-11
...,...,...,...,...,...,...,...,...
9995,TXN_7672686,Coffee,2,2.0,4.0,NaN,UNKNOWN,2023-08-30
9996,TXN_9659401,NaN,3,NaN,3.0,Digital Wallet,NaN,2023-06-02
9997,TXN_5255387,Coffee,4,2.0,8.0,Digital Wallet,NaN,2023-03-02
9998,TXN_7695629,Cookie,3,NaN,3.0,Digital Wallet,NaN,2023-12-02


In [257]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   Transaction ID    10000 non-null  str  
 1   Item              9667 non-null   str  
 2   Quantity          9862 non-null   str  
 3   Price Per Unit    9821 non-null   str  
 4   Total Spent       9827 non-null   str  
 5   Payment Method    7421 non-null   str  
 6   Location          6735 non-null   str  
 7   Transaction Date  9841 non-null   str  
dtypes: str(8)
memory usage: 625.1 KB


In [258]:
df.isnull().sum()

Transaction ID         0
Item                 333
Quantity             138
Price Per Unit       179
Total Spent          173
Payment Method      2579
Location            3265
Transaction Date     159
dtype: int64

In [259]:
df.duplicated().sum()

np.int64(0)

In [260]:
for col in df.columns:
    print(df[col].value_counts(dropna = False))
    print()

Transaction ID
TXN_1961373    1
TXN_4977031    1
TXN_4271903    1
TXN_7034554    1
TXN_3160411    1
              ..
TXN_7672686    1
TXN_9659401    1
TXN_5255387    1
TXN_7695629    1
TXN_6170729    1
Name: count, Length: 10000, dtype: int64

Item
Juice       1171
Coffee      1165
Salad       1148
Cake        1139
Sandwich    1131
Smoothie    1096
Cookie      1092
Tea         1089
UNKNOWN      344
NaN          333
ERROR        292
Name: count, dtype: int64

Quantity
5          2013
2          1974
4          1863
3          1849
1          1822
UNKNOWN     171
ERROR       170
NaN         138
Name: count, dtype: int64

Price Per Unit
3.0        2429
4.0        2331
2.0        1227
5.0        1204
1.0        1143
1.5        1133
ERROR       190
NaN         179
UNKNOWN     164
Name: count, dtype: int64

Total Spent
6.0        979
12.0       939
3.0        930
4.0        923
20.0       746
15.0       734
8.0        677
10.0       524
2.0        497
9.0        479
5.0        468
16.0      

#### Vectorized Transformation

##### Formatting Column names for database insertion

In [261]:
df = df.rename(columns = {'Transaction ID':'Transaction_ID','Price Per Unit':'Price_Per_Unit','Total Spent':'Total_Spent','Payment Method':'Payment_Method','Transaction Date':'Transaction_Date'})

##### Removing Fake Nulls From String Columns

In [262]:
df['Item'] = df['Item'].str.strip().str.title()

In [263]:
df['Payment_Method'] = df['Payment_Method'].str.strip().str.title()

In [264]:
df['Location'] = df['Location'].str.strip().str.title()

In [265]:
junk_values = ['Error','Unknown']

In [266]:
df['Item'] = df['Item'].replace(junk_values, pd.NA)

In [267]:
df['Payment_Method'] = df['Payment_Method'].replace(junk_values, pd.NA)

In [268]:
df['Location'] = df['Location'].replace(junk_values, pd.NA)

##### Memory Optimization: Optimization for low-cardinality columns

In [269]:
df['Item'] = df['Item'].astype('category')

In [270]:
df['Payment_Method'] = df['Payment_Method'].astype('category')

In [271]:
df['Location'] = df['Location'].astype('category')

##### Removing Fake Nulls From Numerical Columns

In [272]:
df['Price_Per_Unit'] = df['Price_Per_Unit'].str.strip().str.title()

In [273]:
df['Quantity'] = df['Quantity'].str.strip().str.title()

In [274]:
df['Total_Spent'] = df['Total_Spent'].str.strip().str.title()

In [275]:
df['Price_Per_Unit'] = df['Price_Per_Unit'].replace(junk_values, pd.NA)

In [276]:
df['Quantity'] = df['Quantity'].replace(junk_values, pd.NA)

In [277]:
df['Total_Spent'] = df['Total_Spent'].replace(junk_values, pd.NA)

##### Numerical Formatting and Type Casting

In [278]:
df['Price_Per_Unit'] = pd.to_numeric(df['Price_Per_Unit'], errors = 'coerce')

In [279]:
df['Quantity'] = pd.to_numeric(df['Quantity'], errors = 'coerce')

In [280]:
df['Total_Spent'] = pd.to_numeric(df['Total_Spent'], errors = 'coerce')

##### Removing Fake Nulls From DateTime Columns

In [281]:
df['Transaction_Date'] = df['Transaction_Date'].str.strip().str.title()

In [282]:
df['Transaction_Date'] = df['Transaction_Date'].replace(junk_values, pd.NA)

##### DateTime Formatting and Type Casting

In [283]:
df['Transaction_Date'] = pd.to_datetime(df['Transaction_Date'], errors = 'coerce', format ='%Y-%m-%d')

#### Filling Null Value(Deterministic Imputation)

* Finding Not Null Values For Price_Per_Unit Of Total_Spent

In [284]:
df['Price_Per_Unit'] = df['Price_Per_Unit'].fillna(df.groupby('Item')['Price_Per_Unit'].transform('first'))

* Finding Not Null Values For Null Values Of Total_Spent

In [285]:
mask_total = df['Total_Spent'].isnull() & df['Quantity'].notnull() & df['Price_Per_Unit'].notnull()

In [286]:
df.loc[mask_total,'Total_Spent'] = df.loc[mask_total,'Quantity'] * df.loc[mask_total,'Price_Per_Unit']

* Finding Not Null Values For Null Values Of Quantity

In [287]:
mask_quantity = df['Quantity'].isnull() & df['Total_Spent'].notnull() & df['Price_Per_Unit'].notnull()

In [288]:
df.loc[mask_quantity, 'Quantity'] = df.loc[mask_quantity, 'Total_Spent'] / df.loc[mask_quantity, 'Price_Per_Unit']

* Finding Not Null Values For Null Values Of Price_Per_Unit For Final Try

In [289]:
mask_price = df['Price_Per_Unit'].isnull() & df['Quantity'].notnull() & df['Total_Spent'].notnull()

In [290]:
df.loc[mask_price, 'Price_Per_Unit'] = df.loc[mask_price, 'Total_Spent'] / df.loc[mask_price, 'Quantity']

##### Dropping Unwanted Nulls From Total_Spent and Quantity

In [291]:
df = df.dropna(subset = ['Total_Spent','Quantity'])

##### Cleaning Process Completed and Review in Process

In [293]:
df.info()

<class 'pandas.DataFrame'>
Index: 9974 entries, 0 to 9999
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   Transaction_ID    9974 non-null   str           
 1   Item              9011 non-null   category      
 2   Quantity          9974 non-null   float64       
 3   Price_Per_Unit    9974 non-null   float64       
 4   Total_Spent       9974 non-null   float64       
 5   Payment_Method    6806 non-null   category      
 6   Location          6022 non-null   category      
 7   Transaction_Date  9514 non-null   datetime64[us]
dtypes: category(3), datetime64[us](1), float64(3), str(1)
memory usage: 496.9 KB


In [294]:
for col in df.columns:
    print(df[col].value_counts(dropna = False))
    print()

Transaction_ID
TXN_1961373    1
TXN_4977031    1
TXN_4271903    1
TXN_7034554    1
TXN_3160411    1
              ..
TXN_7672686    1
TXN_9659401    1
TXN_5255387    1
TXN_7695629    1
TXN_6170729    1
Name: count, Length: 9974, dtype: int64

Item
Juice       1168
Coffee      1160
Salad       1146
Cake        1138
Sandwich    1127
Smoothie    1094
Cookie      1090
Tea         1088
NaN          963
Name: count, dtype: int64

Quantity
5.0    2108
2.0    2055
3.0    1946
4.0    1939
1.0    1926
Name: count, dtype: int64

Price_Per_Unit
3.0    2553
4.0    2448
2.0    1286
5.0    1270
1.0    1211
1.5    1206
Name: count, dtype: int64

Total_Spent
6.0     1019
12.0     996
4.0      969
3.0      968
20.0     788
15.0     766
8.0      720
10.0     542
2.0      518
9.0      509
5.0      496
16.0     466
25.0     268
7.5      250
1.0      249
4.5      238
1.5      212
Name: count, dtype: int64

Payment_Method
NaN               3168
Digital Wallet    2284
Credit Card       2268
Cash              

#### Data Exporting

In [295]:
df.to_csv('clean_cafe_sales.csv', index = False)
print('Export Completed !')

Export Completed !
